# Seminar 1 — Foundations of the Python data stack

**4IZ566 / 4IZ582**

### Learning objectives

Upon completion, students should be able to:

1. Explain Python data types, mutability and object references.
2. Use collections, control flow, functions, exceptions and files.
3. Apply vectorised NumPy operations and distinguish views from copies.
4. Load, select, clean, aggregate and join tabular data in pandas.
5. Construct and interpret statistical graphics with matplotlib.

### Prerequisites and scope

- **Prerequisite:** programming experience in at least one language.
- **Scope:** an introduction to Python tools for data analysis; prior experience with these libraries is not required.
- **[core]:** material covered during the seminar.
- **[extra]:** supplementary material for independent study.

### Contents

| Section | Topic |
|---|---|
| 0 | Jupyter notebooks and execution state |
| 1 | Python types, collections and language constructs |
| 2 | NumPy arrays and vectorised operations |
| 3 | Tabular data processing with pandas |
| 4 | Data visualisation with matplotlib |
| 5 | Review |

## 0 [core] Jupyter notebooks

### Notebook components

| Component | Role |
|---|---|
| Code cell | Executes Python code in the active kernel. |
| Markdown cell | Contains formatted text, equations and documentation. |
| Kernel | Maintains variables and other execution state between cells. |
| `In [n]` | Indicates execution order, which may differ from the displayed cell order. |

### Execution and reproducibility

- `Shift+Enter` executes the current cell and selects the next cell.
- Running cells out of order may introduce dependencies on earlier kernel state.
- **Restart Kernel and Run All Cells** verifies sequential execution from a clean state; successful execution does not by itself establish the correctness of the analysis.

### Interactive tools

`Tab` provides completion, `?` displays help, `%timeit` measures execution time, and `%who` lists defined names.

In [1]:
# Tab completes names; a question mark shows help; %timeit and other "magics" are Jupyter extras.
import math
math.sqrt?

Signature: math.sqrt(x, /)
Docstring: Return the square root of x.
Type:      builtin_function_or_method

In [2]:
# measures execution time
%timeit -n 1000 -r 3 math.sqrt(2.0)
# names defined so far in this kernel
%who                      

244 ns ± 5.17 ns per loop (mean ± std. dev. of 3 runs, 1,000 loops each)
math	 


## 1 [core] Python fundamentals

### 1.1 Data types

#### Objects, names and types

- **Object:** a value with a type that determines its supported operations.
- **Variable:** a name bound to an object; the name may later refer to an object of another type.
- **Type inspection:** `type(value)` returns the type; `isinstance(value, int)` checks membership in a type, including its subclasses.
- **Type-dependent operations:** `22 + 1` performs addition; `"22" + "1"` concatenates strings. Numeric text requires explicit conversion before arithmetic.

#### Overview of built-in types

| Type | Example | Typical use | Mutable? |
|---|---|---|---|
| `int` | `891` | Counts, whole numbers | No |
| `float` | `7.25` | Measurements, approximate decimal numbers | No |
| `bool` | `True`, `False` | Conditions and flags | No |
| `str` | `"Titanic"` | Unicode text | No |
| `NoneType` | `None` | Absence of a value | No |
| `list` | `[22, 38, 26]` | Ordered collection to update | Yes |
| `tuple` | `("Braund", 22)` | Fixed sequence, e.g. a record | No |
| `dict` | `{"age": 22}` | Lookup by key; preserves insertion order | Yes |
| `set` | `{"S", "C"}` | Unique items and set operations | Yes |
| `frozenset` | `frozenset({"S", "C"})` | Immutable set | No |

**Mutability** is the ability to change an object in place. Immutability prevents changes to the object, but does not prevent rebinding a name.

In [3]:
n_passengers = 891
share = 0.384
name = "Titanic"
alive = True
unknown = None
for value in [n_passengers, share, name, alive, unknown]:
    print(repr(value), "->", type(value).__name__)
print(isinstance(n_passengers, int))
print(f"{name}: {n_passengers} passengers, {share:.1%} survived")

891 -> int
0.384 -> float
'Titanic' -> str
True -> bool
None -> NoneType
True
Titanic: 891 passengers, 38.4% survived


### 1.2 Type conversion and scalar operations

#### Explicit conversion

| Expression | Result or behaviour |
|---|---|
| `int("22")` | Integer `22` |
| `float("7.25")` | Floating-point value `7.25` |
| `str(22)` | String `"22"` |
| `int(22.9)` | Integer `22`; truncation towards zero |
| `int("22.9")` | `ValueError`; the string is not an integer representation |

#### Numeric operations

- `/`: division; `//`: floor division; `%`: remainder; `**`: exponentiation.
- Floating-point values approximate real numbers. For computed values, use an appropriate tolerance, for example with `math.isclose`.

#### Truth values and absence

- `None` represents absence of a value; check it with `is None`.
- `None`, numeric zero and empty strings or containers evaluate to false.
- Non-empty strings evaluate to true, including `"False"` and `"0"`.
- Consequently, `bool(text)` does not interpret textual Boolean labels.

In [4]:
raw_age = "22"
print(raw_age + "1", int(raw_age) + 1)       # '221' versus 23
print(7 / 2, 7 // 2, 7 % 2, 2 ** 3)
print(0.1 + 0.2, math.isclose(0.1 + 0.2, 0.3))
print(bool(0), bool([]), bool(""), bool("False"))
print(unknown is None)

# A name can refer to a different type after assignment.
raw_age = int(raw_age)
print(type(raw_age).__name__)

221 23
3.5 3 1 8
0.30000000000000004 True
False False False True
True
int


### 1.3 Strings and collections

#### Selection of a collection type

| Type | Principal operations or properties |
|---|---|
| `str` | Text operations, indexing and slicing |
| `list` | Ordered sequence with insertion, replacement and removal |
| `tuple` | Fixed sequence, unpacking into individual variables |
| `dict` | Lookup and assignment by key; `in` tests keys |
| `set` | Deduplication, membership and set operations; no positional indexing |

#### Indexing and construction

- Strings, lists and tuples use zero-based indexing.
- `[start:stop]` excludes `stop`; `[-1]` selects the final element.
- Lists and tuples may contain values of different types.
- `{}` creates an empty dictionary; `set()` creates an empty set.
- A one-element tuple requires a comma: `(22,)`.

#### Hashability

Dictionary keys and set elements must be **hashable**, with a hash value that remains stable during their lifetime.
Strings and numbers are hashable; lists and dictionaries are not.
A tuple is hashable only when all its elements are hashable.

In [5]:
ages = [22, 38, 26, 35]
record = ("Braund", "Mr", 22)
age_of = {"Braund": 22, "Cumings": 38}
ports = {"S", "C", "Q", "S"}

ages.append(54)
print(ages, len(ages), ages[-1], ages[1:3])
print(name[0], name[:3], name.upper())
age_of["Heikkinen"] = 26
print(age_of["Cumings"], age_of.get("Nobody", "not found"))
print("Braund" in age_of, 22 in age_of.values())
print(len(ports), "C" in ports, ports & {"C", "Q"})
surname, title, age = record                 # tuple unpacking
print(surname, title, age)

[22, 38, 26, 35, 54] 5 54 [38, 26]
T Tit TITANIC
38 not found
True True
3 True {'Q', 'C'}
Braund Mr 22


### 1.4 Mutability, references and identity

#### Assignment and object modification

| Operation | Effect |
|---|---|
| `b = a` | Binds `b` to the object referenced by `a`; no copy is created. |
| `b.append(value)` | Modifies the list; the change is visible through all references to that list. |
| `b = [value]` | Rebinds `b` to a new list; the previous object is not modified. |
| `n += 1` for an integer | Rebinds `n` to the result; the original integer is unchanged. |
| `label.upper()` | Returns a string result without modifying `label`. |

#### Equality and identity

- **Equality (`==`):** compares values.
- **Identity (`is`):** checks whether operands refer to the same object.
- Use `==` for numeric and text comparisons; use `is None` to test for absence.

In [6]:
n = 22
previous_n = n
n += 1
print(n, previous_n)                         # 23 22: the original integer did not change

label = "titanic"
upper_label = label.upper()
print(label, upper_label)                    # titanic TITANIC
# label[0] = "T" would raise TypeError: strings do not support item assignment.

a = [22, 38]
b = a
b[0] = 23                                   # changes the shared list
print(a, b, a is b)                          # [23, 38] [23, 38] True
b = [99]                                    # rebinds b only
print(a, b)                                 # [23, 38] [99]

23 22
titanic TITANIC
[23, 38] [23, 38] True
[23, 38] [99]


### 1.5 Copying and in-place operations

- **List copy:** `a.copy()` or `a[:]` creates a new outer list. Replacing its elements does not modify `a`.
- **In-place operation:** `list.append()` changes the list and returns `None`. Assigning this result to the list variable loses that variable's reference to the list.
- **Function arguments:** a function receives references to objects. Mutating a list parameter also affects the list referenced by the caller.

In [7]:
a = [22, 38]
b = a.copy()
print(a == b, a is b)                        # True False: equal values, distinct lists
b[0] = 99
result = b.append(54)
print(a, b, result)                          # [22, 38] [99, 38, 54] None

True False
[22, 38] [99, 38, 54] None


### 1.6 Exercise: references and copies

Before executing the following cell:

1. Determine the final values of `original`, `alias`, `independent` and `text`.
2. Identify the names that refer to the same object.
3. Modify the assignment to `alias` so that appending to it leaves `original` unchanged.

In [8]:
original = [1, 2]
alias = original
independent = original.copy()
alias.append(3)
independent[0] = 99
text = "python"
text.upper()
print(original, alias, independent, text)

[1, 2, 3] [1, 2, 3] [99, 2] python


<details>
<summary>Solution and explanation</summary>

| Name | Final value | Explanation |
|---|---|---|
| `original` | `[1, 2, 3]` | References the list modified through `alias`. |
| `alias` | `[1, 2, 3]` | References the same list as `original`. |
| `independent` | `[99, 2]` | References a separate list created by `.copy()`. |
| `text` | `"python"` | The result of `.upper()` was not assigned. |

Use `alias = original.copy()` before the append to preserve `original`.
Use `text = text.upper()` to retain the uppercase result.

</details>

### 1.7 [extra] Nested objects and copy depth

| Concept | Behaviour |
|---|---|
| Shallow copy: `.copy()` | Creates an outer container but retains references to its elements. |
| Deep copy: `copy.deepcopy()` | Recursively copies nested contents where supported. |
| Tuple immutability | Prevents replacement of the tuple's element references; mutable elements may still change. |

In [9]:
from copy import deepcopy

rows = [["Braund", 22], ["Cumings", 38]]
shallow = rows.copy()
deep = deepcopy(rows)
shallow[0][1] = 23
print(rows[0], deep[0])                      # ['Braund', 23] ['Braund', 22]

passenger = ("Braund", ["ticket", "bag"])
passenger[1].append("coat")                  # the inner list is mutable
print(passenger)
# passenger[1] = [] would raise TypeError: tuple elements cannot be reassigned.

['Braund', 23] ['Braund', 22]
('Braund', ['ticket', 'bag', 'coat'])


### 1.8 [extra] Mutable default arguments

- **Evaluation:** default arguments are evaluated when a function is defined.
- **Consequence:** calls that omit a list argument share the same default list.
- **Recommended pattern:** use `None` as the default and create a fresh list inside the function.

In [10]:
def add_item(item, basket=[]):               # shared default list, created at function definition
    basket.append(item)
    return basket

print(add_item("hat"))                       # ['hat']
print(add_item("scarf"))                     # ['hat', 'scarf']

def add_item_fixed(item, basket=None):
    basket = [] if basket is None else basket
    basket.append(item)
    return basket

print(add_item_fixed("hat"), add_item_fixed("scarf"))

['hat']
['hat', 'scarf']
['hat'] ['scarf']


### 1.9 Control flow and comprehensions

| Construct | Purpose |
|---|---|
| `for` and tuple unpacking | Iterate over records and name their fields. |
| `if` / `elif` / `else` | Select a branch according to a condition. |
| List and dictionary comprehensions | Construct collections by transformation and filtering. |
| `enumerate` and `zip` | Associate elements with positions or iterate over sequences together. |
| `while` | Repeat a block while a condition holds. |

In [11]:
records = [
    ("Prague",  1_384_732, 496.2),
    ("Brno",      396_683, 230.2),
    ("Ostrava",   284_765, 214.2),
    ("Plzen",     186_990, 137.7),
]

# 1) Tuple unpacking in a loop — no indexing by position; if/elif/else inside.
for city, population, area in records:
    size = "large" if population > 300_000 else "small"
    print(f"{city:<10} {population/area:8.1f} inhabitants/km2  ({size})")

Prague       2790.7 inhabitants/km2  (large)
Brno         1723.2 inhabitants/km2  (large)
Ostrava      1329.4 inhabitants/km2  (small)
Plzen        1358.0 inhabitants/km2  (small)


In [12]:
# 2) List and dict comprehensions replace append-loops.
density = {city: population / area for city, population, area in records}
big = [city for city, pop, _ in records if pop > 300_000]
print(density)
print(big)

{'Prague': 2790.6731156791616, 'Brno': 1723.210251954822, 'Ostrava': 1329.435107376284, 'Plzen': 1357.9520697167757}
['Prague', 'Brno']


In [13]:
# 3) enumerate and zip provide positions and paired values directly.
names = [r[0] for r in records]
sizes = [r[1] for r in records]
for rank, (name_, size_) in enumerate(zip(names, sizes), start=1):
    print(rank, name_, size_)

# 4) while — for loops whose length you do not know in advance
total, k = 0, 0
while total < 2_000_000:
    total += sizes[k]; k += 1
print("the first", k, "cities together exceed two million inhabitants")

1 Prague 1384732
2 Brno 396683
3 Ostrava 284765
4 Plzen 186990
the first 3 cities together exceed two million inhabitants


### 1.10 Functions

- **Parameters:** define inputs, optionally with default values.
- **Return value:** a function returns one object; a tuple can group several results.
- **Functions as objects:** functions can be assigned to names and passed as arguments, for example to `sorted(key=...)`.
- **Lambda expression:** defines a function using a single expression.

In [16]:
sample = [{"name": "Braund", "age": 22, "survived": 0}, {"name": "Cumings", "age": 38, "survived": 1},
          {"name": "Heikkinen", "age": 26, "survived": 1}, {"name": "Allen", "age": 35, "survived": 0}]

def survival_rate(people, min_age=0):
    # share of survivors among people at least min_age years old, and how many were counted
    eligible = [r for r in people if r["age"] >= min_age]
    if not eligible:
        return 0.0, 0
    return sum(r["survived"] for r in eligible) / len(eligible), len(eligible)

rate, n = survival_rate(sample, min_age=30)           # a returned tuple unpacks into two names
print(f"{rate:.2f} of {n} passengers aged 30+ survived")

oldest_first = sorted(sample, key=lambda r: r["age"], reverse=True)     # a one-line function as the key
print([r["name"] for r in oldest_first])
print(sorted(["Brno", "Plzen", "As"], key=len))                          # a built-in function as the key

0.50 of 2 passengers aged 30+ survived
['Cumings', 'Allen', 'Heikkinen', 'Braund']
['As', 'Brno', 'Plzen']


### 1.11 Exception handling

| Construct | Purpose |
|---|---|
| `try` / `except` | Handle an anticipated exception, such as invalid numeric input. |
| `except ValueError` | Restrict handling to the specified exception type. |
| `raise ValueError("message")` | Reject input that violates a function's requirements. |

Exception types and messages document the conditions under which a function rejects its input.

In [14]:
def parse_age(text):
    try:
        return float(text)
    except ValueError:            # handle invalid numeric input; propagate other exceptions
        return None

print(parse_age("22"), parse_age("22.5"), parse_age("n/a"))

def checked_fare(fare):
    if fare < 0:
        raise ValueError("fare must be non-negative")
    return fare

try:
    checked_fare(-7.25)
except ValueError as e:
    print("caught:", e)

22.0 22.5 None
caught: fare must be non-negative


### 1.12 Text files

- `with open(...)` manages a file resource and closes it when the block exits, including after an exception.
- Specify an encoding, for example `encoding="utf-8"`, when reading or writing text.
- Typical applications include logs, configuration and small text outputs.
- Section 3 introduces `pandas.read_csv` for structured tabular files.

In [17]:
with open("passengers_sample.txt", "w", encoding="utf-8") as fh:
    for r in sample:
        fh.write(f"{r['name']},{r['age']},{r['survived']}")
        fh.write(chr(10))                                   # newline

with open("passengers_sample.txt", encoding="utf-8") as fh:
    lines = [line.rstrip() for line in fh]
print(len(lines), "lines;", "first:", lines[0])

4 lines; first: Braund,22,0


### 1.13 Objects and classes

| Concept | Definition | Example |
|---|---|---|
| Object | A value with a type and associated behaviour | A string or a passenger instance |
| Attribute | Data accessible through an object | `p.age` |
| Method | A function accessed through an object or class | `p.is_adult()` |
| Class | Defines the structure and behaviour of its instances | `Passenger` |

The following class groups passenger attributes with a method for determining adulthood.

In [18]:
class Passenger:
    # passenger attributes and instance methods

    def __init__(self, name, age, survived):     # runs when an object is created
        self.name = name                          # attributes live on the instance, via self
        self.age = age
        self.survived = survived

    def is_adult(self):                           # a method: a function that receives the instance as self
        return self.age >= 18

    def __repr__(self):                           # how the object prints itself
        return f"Passenger({self.name!r}, {self.age}, survived={self.survived})"

p = Passenger("Braund, Mr. Owen Harris", 22, False)
print(p)                                  # uses __repr__
print(p.age, p.is_adult())                # attribute, method
print(type(p), isinstance(p, Passenger))

people = [Passenger(r["name"], r["age"], bool(r["survived"])) for r in sample]
print(sum(x.survived for x in people), "of", len(people), "survived;", [x.name for x in people if x.is_adult()][:2])

Passenger('Braund, Mr. Owen Harris', 22, survived=False)
22 True
<class '__main__.Passenger'> True
2 of 4 survived; ['Braund', 'Cumings']


#### Inheritance

A subclass inherits behaviour from a parent class and may override selected methods.
The following illustrative subclass overrides `is_adult()` while retaining the inherited attributes.

In [19]:
class Crew(Passenger):                     # a Crew member IS a Passenger, with one rule changed
    def is_adult(self):
        return True                        # illustrative override for this subclass

c = Crew("Andrews, Mr. Thomas", 39, False)
print(c, "| adult:", c.is_adult(), "| still a Passenger:", isinstance(c, Passenger))

Passenger('Andrews, Mr. Thomas', 39, survived=False) | adult: True | still a Passenger: True


#### Object interfaces in data analysis

| Object | Attribute examples | Method examples |
|---|---|---|
| pandas `DataFrame` | `shape` | `head()` |
| NumPy array | `dtype`, `shape` | `mean()` |
| Fitted scikit-learn classifier | `coef_` or `feature_importances_`, depending on the estimator | `fit()`, `predict()` |

**Inspection:** `type(obj)` identifies the type; `dir(obj)` lists attribute names; `help(obj.method)` or `obj.method?` displays documentation.

**Shared interfaces:** seminar 3 uses a simulated client with the same method interface as the remote client, allowing the calling code to work with either implementation.

In [ ]:
print([name for name in dir(p) if not name.startswith("_")])      # the public attributes and methods

In [ ]:
# Generators compute lazily and can be consumed once. Useful when the data does not fit in
# memory; wrapping one in list() gives the values back but gives the laziness away.
def running_total(values):
    total = 0
    for v in values:
        total += v
        yield total

gen = running_total(sizes)
print(next(gen), next(gen))      # two values computed so far
print(list(gen))                 # the rest; the generator is now exhausted

### 1.14 [extra] File paths

- `pathlib.Path` provides path operations across operating systems.
- The `/` operator combines path components without manually inserting separators.
- `Path.cwd()` returns the kernel's working directory, which may differ from the notebook's directory.

In [ ]:
from pathlib import Path

here = Path.cwd()
print(here)
print(here / "data" / "example.csv")   # no slashes hardcoded
# pandas accepts Path objects directly: pd.read_csv(here / "data" / "example.csv")

## 2 [core] NumPy arrays and vectorisation

### 2.1 Array representation

| Property | Meaning |
|---|---|
| `dtype` | Common element type |
| `shape` | Length along each dimension |
| Strides | Byte offsets used to move along each axis |
| Memory layout | May be contiguous or non-contiguous, for example after transposition or slicing |

### Vectorised computation

Vectorised operations execute loops over typed array data in compiled code, reducing Python-level iteration overhead.
The following benchmark compares an explicit Python loop with a NumPy operation; the speed difference depends on the operation, array size and environment.

In [20]:
import numpy as np

rng = np.random.default_rng(42)     # always seed: results must be reproducible
x = rng.standard_normal(1_000_000)
print(x.dtype, x.shape, x.nbytes / 1e6, "MB")

m = np.arange(12).reshape(3, 4)
print("fresh array contiguous:", m.flags["C_CONTIGUOUS"],
      "| its transpose:", m.T.flags["C_CONTIGUOUS"])

float64 (1000000,) 8.0 MB
fresh array contiguous: True | its transpose: False


In [21]:
def sum_of_squares_loop(values):
    total = 0.0
    for v in values:
        total += v * v
    return total

%timeit -n 3 -r 3 sum_of_squares_loop(x)

195 ms ± 4.65 ms per loop (mean ± std. dev. of 3 runs, 3 loops each)


In [22]:
%timeit -n 3 -r 3 np.sum(x ** 2)

2.56 ms ± 1.58 ms per loop (mean ± std. dev. of 3 runs, 3 loops each)


### 2.2 Broadcasting

**Compatibility rules:**

1. Compare shapes from the trailing dimension.
2. Corresponding dimensions must be equal, or one must have length 1.
3. Missing leading dimensions are treated as having length 1.

Broadcasting applies compatible values across dimensions without requiring explicit repetition of the input data.
The same shape rules also occur in PyTorch.

**Exercise:** for `a.shape == (4, 3)` and `col_means.shape == (3,)`, determine the shape of `a - col_means` and the dimension across which the means are applied.

In [ ]:
a = np.arange(12, dtype=float).reshape(4, 3)
col_means = a.mean(axis=0)          # shape (3,)
print(a)
print("column means:", col_means)
print("centred:")
print(a - col_means)                # (4,3) - (3,) -> (4,3)

In [ ]:
# Broadcasting fails loudly when the rule cannot be applied. Read the shapes in the message.
try:
    a + np.arange(4)
except ValueError as e:
    print("ValueError:", e)

In [ ]:
# Fix it by making the intent explicit — give the second operand shape (4,1).
print(a + np.arange(4)[:, None])

#### Semantic errors with compatible shapes

- A square matrix of shape `(3, 3)` and a vector of shape `(3,)` are broadcast-compatible.
- Subtracting a vector of row means in this form associates each mean with a column position, rather than its corresponding row.
- To centre each row, retain the singleton column dimension: `(3, 1)`.

**Exercise:** predict the row means printed by the following cell.

In [ ]:
sq = np.arange(9, dtype=float).reshape(3, 3)
row_means = sq.mean(axis=1)                       # shape (3,): one mean per ROW
print("rows minus their own mean should be zero-mean per row:")
print((sq - row_means[:, None]).mean(axis=1))     # correct: explicit column vector
print("but sq - row_means broadcasts along the wrong axis and nobody complains:")
print((sq - row_means).mean(axis=1))              # wrong, and silent

### 2.3 Views and copies

| Selection | Result | Effect of modifying the result |
|---|---|---|
| Basic slicing | View sharing the original memory | Modifies the original array |
| Advanced indexing with integer indices or a Boolean mask | Copy | Does not modify the original array |
| `.copy()` | Explicit copy | Does not modify the original array |

These examples use numeric arrays. `np.shares_memory` checks whether arrays share memory.

In [ ]:
base = np.arange(10)

view = base[2:5]          # slice -> view
view[:] = -1
print("after writing into a slice:", base)

base = np.arange(10)
copy = base[[2, 3, 4]]    # fancy indexing -> copy
copy[:] = -1
print("after writing into fancy-indexed result:", base)

In [ ]:
# When in doubt, ask NumPy. And when you need independence, say so with .copy().
s = base[2:5]
print("slice shares memory with base :", np.shares_memory(s, base))
print(".copy() shares memory with base:", np.shares_memory(base[2:5].copy(), base))

### 2.4 Masks and axes

- **Boolean mask:** selects elements satisfying a condition; it can also be used with `np.where` for conditional computation.
- **Reduction along `axis=0`:** for a matrix, aggregates rows and produces one result per column.
- **Reduction along `axis=1`:** for a matrix, aggregates columns and produces one result per row.

The following examples also demonstrate sorting indices and changing array shape.

In [ ]:
m = rng.integers(0, 100, size=(4, 5))
print(m)
print("sum per column (axis=0):", m.sum(axis=0))
print("sum per row    (axis=1):", m.sum(axis=1))

mask = m > 50
print("values > 50:", mask.sum(), "| their mean:", m[mask].mean().round(1))
print(np.where(m > 50, 1, 0))                                    # vectorised if/else
order = np.argsort(m[0])
print("argsort of row 0:", order, "-> sorted:", m[0][order])       # argsort gives positions, not values
print(m.astype(float).dtype, "|", m.reshape(2, 10).shape, "|", m.T.shape, "|", np.concatenate([m[:1], m[-1:]]).shape)

## 3 [core] Tabular data processing with pandas

### Data model and dataset

- **`DataFrame`:** a two-dimensional labelled table whose columns are `Series` objects sharing a row index.
- **Dataset:** 891 Titanic passengers, with survival, class, sex, age, fare, cabin and embarkation information.
- **Workflow:** load → select → clean → explore → aggregate → join; reshaping is included as supplementary material.

### 3.1 Loading CSV data

`pd.read_csv` reads column names from the header, infers column types and recognises missing-value markers.
The example uses `data/titanic.csv`; the `TITANIC_CSV` environment variable can specify an alternative path.

In [23]:
import os
import pandas as pd
from pathlib import Path

csv_path = Path("data") / "titanic.csv"
if not csv_path.exists():
    csv_path = Path(os.environ.get("TITANIC_CSV", csv_path))
titanic = pd.read_csv(csv_path)
print(titanic.shape, "rows x columns")
titanic.head()

(891, 12) rows x columns


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


#### Import parameters and initial inspection

| Parameter or method | Purpose |
|---|---|
| `sep=";"` | Specify a semicolon delimiter. |
| `encoding="cp1250"` | Read text encoded in Windows Central European encoding. |
| `na_values=["n/a", "-"]` | Recognise additional missing-value markers. |
| `parse_dates=[...]` | Parse selected columns as dates. |
| `usecols=[...]` | Load selected columns only. |
| `df.info()` | Inspect inferred types and non-missing counts. |
| `df.to_csv("out.csv", index=False)` | Export without writing the row index as an additional column. |

Select import parameters according to the source file's format and encoding.

In [ ]:
titanic.info()

In [ ]:
print("missing values per column:")
print(titanic.isna().sum())

**Initial missing-value counts:** `Age`: 177; `Cabin`: 687; `Embarked`: 2.
Treatment of these values depends on the analytical objective.

### 3.2 Selection by label and position

| Indexer | Selection basis | Slice endpoint |
|---|---|---|
| `.loc` | Row and column labels | Included |
| `.iloc` | Integer positions | Excluded, as in Python sequence slicing |

In [ ]:
print(titanic.loc[titanic["Sex"] == "female", ["Name", "Pclass", "Survived"]].head())
print(titanic.iloc[:3, :4])
print("loc 0:2 gives", len(titanic.loc[0:2]), "rows; iloc 0:2 gives", len(titanic.iloc[0:2]))

In [ ]:
first_class_women = titanic.loc[(titanic["Sex"] == "female") & (titanic["Pclass"] == 1)]
print(len(first_class_women), "women in first class,", round(first_class_women["Survived"].mean(), 3), "of them survived")

### 3.3 Index alignment in arithmetic

- Arithmetic between `Series` aligns values by **index label**, not by position.
- Different labels may introduce missing values even when both Series have the same length.
- Verify index compatibility after filtering, sorting or reindexing.

In [ ]:
a = pd.Series([1, 2, 3], index=[0, 1, 2])
b = pd.Series([10, 20, 30], index=[1, 2, 3])
print(a + b)                                   # NaN at 0 and 3: aligned by label, not by position
print(a.to_numpy() + b.to_numpy())             # positional, if that is what you meant

fares = titanic["Fare"].head(3)
print(fares + fares.sort_values(ascending=False))   # same rows in another order: still 2x each fare

### 3.4 Assignment with `.loc`

**Chained assignment:** `df[mask]["col"] = value` operates through an intermediate object.

| Version | Behaviour |
|---|---|
| pandas 2 | The intermediate object may be a copy; assignment may fail to update the original, potentially with a warning. |
| pandas 3 | Under Copy-on-Write, chained assignment does not update the original and produces a `ChainedAssignmentError` warning. |

**Direct assignment:** use `df.loc[row_selector, column] = value` to specify rows and columns in a single operation.

In [ ]:
import warnings

demo = titanic.head(5).copy()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")                       # both pandas 2 and 3 warn here
    demo[demo["Sex"] == "male"]["Age"] = 0                # chained: does NOT update demo
print("after chained assignment, ages set to 0:", (demo["Age"] == 0).sum())

demo.loc[demo["Sex"] == "male", "Age"] = 0                # the one correct form
print("after .loc assignment, ages set to 0   :", (demo["Age"] == 0).sum())
print("pandas", pd.__version__)

### 3.5 Text processing and type conversion

#### String columns

- pandas 3 uses a dedicated string dtype by default for text, with PyArrow backing where available.
- `dtype == object` is therefore not a reliable text-type check; use an appropriate check such as `pd.api.types.is_string_dtype`.
- The `.str` accessor provides elementwise text operations.

#### Extraction example

The `Name` column follows the pattern `"Surname, Title. First names"`.
A regular expression extracts the title as a categorical variable.

In [ ]:
titanic["Title"] = titanic["Name"].str.extract(r",\s*([^.]+)\.")[0].str.strip()
print(titanic["Title"].value_counts().head(6))

#### Conversion of numeric text

1. Normalise textual representations as needed.
2. Apply `pd.to_numeric(errors="coerce")`; unparseable values become missing.
3. Inspect the missing values introduced by conversion.
4. Choose a treatment appropriate to the analysis.

The example works on a copy to preserve the imported data and resets the index after removing rows.

In [ ]:
messy = pd.DataFrame({
    "city": [" Prague", "brno ", "PRAGUE", "Ostrava", " Brno"],
    "amount": ["1200.5", "800", "n/a", "", "450.25"],
})
tidy = messy.copy()
tidy["city"] = tidy["city"].str.strip().str.capitalize()
tidy["amount"] = pd.to_numeric(tidy["amount"], errors="coerce")
print(tidy)
print("missing after coercion:", tidy["amount"].isna().sum())
tidy = tidy.dropna(subset=["amount"]).reset_index(drop=True)
print(tidy)
print("messy is untouched:", messy["city"].iloc[0] == " Prague")

### 3.6 Missing-value treatment

| Variable | Missingness | Illustrative treatment |
|---|---|---|
| `Cabin` | Approximately 77% | Derive a flag indicating whether cabin information is recorded. |
| `Embarked` | 2 records | Remove affected rows or impute the most frequent category. |
| `Age` | Approximately 20% | Impute a median within groups defined by passenger class and sex. |

- `groupby(...).transform(...)` returns values aligned with the original rows, suitable for use with `fillna`.
- These strategies are illustrative choices; their suitability depends on the data and analytical objective.
- For predictive modelling, estimate imputation values from training data only and apply them to held-out data. Pipelines and data leakage are covered in seminar 2.

In [ ]:
filled = titanic.copy()
filled["HasCabin"] = filled["Cabin"].notna()
filled = filled.dropna(subset=["Embarked"]).reset_index(drop=True)
filled["Age"] = filled["Age"].fillna(filled.groupby(["Pclass", "Sex"])["Age"].transform("median"))

print("missing ages before:", titanic["Age"].isna().sum(), "| after group-wise imputation:", filled["Age"].isna().sum())
print("rows:", len(titanic), "->", len(filled), "(two passengers without a port dropped)")
print(filled.groupby(["Pclass", "Sex"])["Age"].median().round(1))

### 3.7 Exploratory data analysis

Exploratory data analysis should report:

1. Dataset dimensions and variable types.
2. Missing-value frequencies.
3. Target-class frequencies.
4. Descriptive statistics and at least two relevant visualisations.

**Example:** approximately 38% of passengers survived. Predicting non-survival for every passenger would achieve approximately 62% accuracy; this provides a baseline for interpreting model performance.

In [ ]:
print(titanic.describe().round(2))
print()
print("class balance of the target (Survived):")
print(titanic["Survived"].value_counts(normalize=True).round(3))
print()
print("correlations between numeric columns:")
print(titanic[["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"]].corr().round(2))

### 3.8 Grouped aggregation

`groupby` partitions data into groups, applies an operation and combines the results.

| Operation or option | Meaning |
|---|---|
| `.count()` | Count non-missing values. |
| `.size()` | Count rows, including those with missing values. |
| `.agg(...)` | Specify one or more named aggregations. |
| `dropna=False` | Retain groups with missing grouping keys. |

For `Age`, total row counts and non-missing counts differ by 177.

In [ ]:
print(titanic.groupby("Sex")["Survived"].mean().round(3))
print()
by_class_sex = titanic.groupby(["Pclass", "Sex"]).agg(
    survival_rate=("Survived", "mean"),
    passengers=("Survived", "size"),        # rows
    age_known=("Age", "count"),             # non-missing ages
    median_fare=("Fare", "median"),
).round(3)
by_class_sex

### 3.9 Table joins

The example enriches passenger records with port names using a left join on the embarkation code.

#### Join validation

- **Row count:** an increase after a left join indicates multiple right-side matches for at least one left-side row.
- **Key cardinality:** `validate="many_to_one"` checks that the right-side key is unique.
- **Match status:** `indicator=True` identifies matched and unmatched records.

Duplicate matching keys can multiply rows and alter subsequent aggregates.

In [ ]:
ports = pd.DataFrame({"Embarked": ["C", "Q", "S"],
                      "Port": ["Cherbourg", "Queenstown", "Southampton"]})

merged = titanic.merge(ports, on="Embarked", how="left", validate="many_to_one", indicator=True)
print(merged.shape, "| rows preserved:", len(merged) == len(titanic))
print(merged["_merge"].value_counts())          # the two passengers without a port stay unmatched
merged.groupby("Port")["Survived"].agg(["size", "mean"]).round(3)

In [ ]:
# What happens when the right-hand key is NOT unique:
dup = pd.concat([ports, ports.iloc[[0]]], ignore_index=True)
try:
    titanic.merge(dup, on="Embarked", how="left", validate="many_to_one")
except pd.errors.MergeError as e:
    print("MergeError:", str(e)[:90], "...")
print("without validate= you would silently get", len(titanic.merge(dup, on="Embarked", how="left")),
      "rows instead of", len(titanic))

### 3.10 [extra] Reshaping

| Method | Purpose |
|---|---|
| `pivot_table` | Produce a wider table, aggregating duplicate combinations; the default aggregation is the mean. |
| `pivot` | Produce a wider table without aggregation; duplicate combinations raise an error. |
| `melt` | Convert selected columns into variable–value rows. |

In [ ]:
wide = titanic.pivot_table(index="Pclass", columns="Sex", values="Survived", aggfunc="mean").round(3)
print(wide)
print()
print(wide.reset_index().melt(id_vars="Pclass", value_name="survival_rate"))

### 3.11 [extra] Recoding, sorting and renaming

| Operation | Purpose |
|---|---|
| `map(mapping)` | Recode values using a dictionary. |
| `.str[0]` | Extract the first character while preserving missing values. |
| `sort_values(...)` | Sort by one or more columns, with specified directions. |
| `rename(...)` | Rename labels. |

Prefer built-in vectorised operations when available. Row-wise `apply` executes a Python function for each row and can incur substantial overhead.

In [ ]:
recoded = titanic.copy()
recoded["SexCode"] = recoded["Sex"].map({"male": 0, "female": 1})
recoded["Deck"] = recoded["Cabin"].str[0]
print(recoded.sort_values(["Pclass", "Fare"], ascending=[True, False])[["Pclass", "Fare", "Name"]].head(3))
print(recoded.rename(columns={"SibSp": "Siblings", "Parch": "ParentsChildren"}).columns.tolist()[:8])
print(recoded["Deck"].value_counts(dropna=False).head(4))

### 3.12 [extra] SQL data access

1. Establish a database connection; this example uses in-memory SQLite.
2. Write a DataFrame to a table with `to_sql`.
3. Execute a SQL query and retrieve a DataFrame with `read_sql`.
4. Close the connection after use.

In [ ]:
import sqlite3

con = sqlite3.connect(":memory:")
titanic.to_sql("passengers", con, index=False)
query = "SELECT Pclass, AVG(Survived) AS survival_rate, COUNT(*) AS n FROM passengers GROUP BY Pclass"
print(pd.read_sql(query, con))
con.close()

## 4 [core] Data visualisation with matplotlib

### 4.1 Figures and axes

| Interface | Behaviour |
|---|---|
| Implicit: `plt.plot`, `plt.xlabel` | Operates on the current axes maintained by pyplot. |
| Explicit: `fig, ax = plt.subplots()` | Uses explicit references to a `Figure` and its `Axes`. |

The examples use the explicit interface to associate each operation with a specific plot and support multi-panel figures.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(titanic["Age"].dropna(), bins=30, edgecolor="white")
ax.set_xlabel("Age [years]")
ax.set_ylabel("Number of passengers")
ax.set_title("Age of passengers (714 known of 891)")
fig.tight_layout()

### 4.2 Plot selection and labelling

| Variable or objective | Suitable representation |
|---|---|
| Category frequencies or summaries | Bar chart |
| Distribution of a numeric variable | Histogram or box plot |
| Strongly skewed positive values | Consider a logarithmic axis |
| Record identifier | Use as a label or key, rather than treating it as a measured quantity |

Label axes, include units where applicable, and use titles that identify the quantities being compared.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

titanic.groupby("Pclass")["Survived"].mean().plot.bar(ax=axes[0], color="steelblue")
axes[0].set_xlabel("Passenger class")
axes[0].set_ylabel("Survival rate")
axes[0].set_title("Survival by class")
axes[0].set_ylim(0, 1)

fares = [g["Fare"].to_numpy() for _, g in titanic.groupby("Pclass")]
axes[1].boxplot(fares, tick_labels=["1st", "2nd", "3rd"])
axes[1].set_yscale("log")
axes[1].set_xlabel("Passenger class")
axes[1].set_ylabel("Fare [£, log scale]")
axes[1].set_title("Fare by class")

fig.tight_layout()

In [ ]:
# pandas delegates to matplotlib and returns the Axes, so both APIs mix freely.
top = titanic["Title"].value_counts().head(5).index
ax = titanic[titanic["Title"].isin(top)].groupby("Title")["Survived"].mean().sort_values().plot.barh(figsize=(6, 3))
ax.set_xlabel("Survival rate")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.figure.tight_layout()

### 4.3 Ordered data, annotations and export

- **Line plot:** connects observations with a meaningful order; the example uses age groups.
- **`annotate`:** associates explanatory text with a selected location.
- **`savefig`:** exports the figure, for example as PNG at a specified resolution or as PDF/SVG for vector output.

In [ ]:
age_group = pd.cut(titanic["Age"], bins=[0, 12, 18, 30, 45, 60, 80])
rate_by_age = titanic.groupby(age_group, observed=True)["Survived"].mean()

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(range(len(rate_by_age)), rate_by_age.to_numpy(), marker="o", label="survival rate")
ax.set_xticks(range(len(rate_by_age)), [str(b) for b in rate_by_age.index], rotation=20)
ax.axhline(titanic["Survived"].mean(), ls="--", c="grey", label="all passengers")
ax.annotate("children first", xy=(0, rate_by_age.iloc[0]), xytext=(1.3, 0.85),
            arrowprops=dict(arrowstyle="->"))
ax.set_xlabel("Age group [years]"); ax.set_ylabel("Survival rate"); ax.set_ylim(0, 1); ax.legend()
fig.tight_layout()
fig.savefig("survival_by_age.png", dpi=150)
print("saved survival_by_age.png")

## 5 Review

### Key concepts

| Area | Concepts to review |
|---|---|
| Python types | Explicit conversion, mutability, equality and identity |
| Object references | Assignment, shallow copying and nested mutable objects |
| Language constructs | Collections, control flow, functions, exceptions and classes |
| NumPy | Shape, dtype, broadcasting, axes, views and copies |
| pandas | Index alignment, `.loc`, missing values, aggregation and join validation |
| matplotlib | Figures, axes, plot selection and informative labels |

### Review questions

1. How does rebinding a variable differ from modifying a mutable object?
2. When does a shallow copy retain shared state?
3. How do array shapes determine broadcasting compatibility?
4. Why can a join increase the number of rows?

### Sources

- Wes McKinney, *Python for Data Analysis*, 3rd edition: <https://wesmckinney.com/book/>, chapters 4–9.
- Titanic passenger dataset: `data/titanic.csv`, Kaggle version of the Encyclopedia Titanica data.